In [3]:
import joblib
import os
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

In [ ]:
os.makedirs("saved_bottleneck_models", exist_ok=True)

print("Loading dataset...")

train_df = pd.read_csv("Dataset/Train.csv")

print("Dataset Shape:", train_df.shape)


Loading dataset...
Dataset Shape: (38373, 10)


In [ ]:
train_df["TimeStamp"] = pd.to_datetime(
    train_df["TimeStamp"],
    format="%d/%m/%y %H:%M",
    errors="coerce"
)

train_df = train_df.dropna(
    subset=["TimeStamp"]
)

In [ ]:
train_df["Hour"] = train_df["TimeStamp"].dt.hour

train_df["Day"] = train_df["TimeStamp"].dt.day

train_df["Month"] = train_df["TimeStamp"].dt.month

train_df["Year"] = train_df["TimeStamp"].dt.year

train_df["Weekday"] = train_df["TimeStamp"].dt.weekday

train_df["Date_Only"] = train_df["TimeStamp"].dt.date

In [ ]:
train_df["IsWeekend"] = (
    train_df["Weekday"] >= 5
).astype(int)


if "Holiday" in train_df.columns:

    train_df["IsHoliday"] = (
        train_df["Holiday"].notnull()
    ).astype(int)


In [ ]:
train_df["Traffic_Vol"] = pd.to_numeric(
    train_df["Traffic_Vol"],
    errors="coerce"
)

train_df = train_df.dropna(
    subset=["Traffic_Vol"]
)

train_df = train_df.ffill().bfill()

In [ ]:
q33 = train_df[
    "Traffic_Vol"
].quantile(0.33)

q66 = train_df[
    "Traffic_Vol"
].quantile(0.66)


def get_traffic_level(volume):

    if volume <= q33:

        return "Low Traffic"

    elif volume <= q66:

        return "Moderate Traffic"

    else:

        return "High Traffic"


train_df[
    "Traffic_Level"
] = train_df[
    "Traffic_Vol"
].apply(
    get_traffic_level
)


print("\nTraffic Thresholds")

print("Low <=", q33)

print("Moderate <=", q66)

print("High >", q66)


Traffic Thresholds
Low <= 2231.0
Moderate <= 4575.0
High > 4575.0


In [ ]:
analysis_features = [

    "Hour",

    "Day",

    "Month",

    "Year",

    "Weekday",

    "IsWeekend",

    "IsHoliday",

    "Weather",

    "Weather_Desc",

    "Temperature",

    "Rainfall_last_hour",

    "Snowfall_last_hour",

    "Cloud_Cover"

]


# Keep only features existing in dataset

analysis_features = [

    feature

    for feature in analysis_features

    if feature in train_df.columns

]

In [ ]:
dbscan_models = {}

feature_encodings = {}

feature_cluster_info = {}


for feature in analysis_features:

    print(
        f"\nTraining DBSCAN for: {feature}"
    )


    temp_df = train_df.copy()

    if temp_df[feature].dtype == "object":

        codes, uniques = pd.factorize(
            temp_df[feature]
        )


        temp_df[
            "Feature_Value"
        ] = codes


        feature_encodings[
            feature
        ] = {

            value: index

            for index, value

            in enumerate(uniques)

        }

    else:

        temp_df[
            "Feature_Value"
        ] = temp_df[
            feature
        ]

    X = temp_df[

        [
            "Feature_Value",
            "Traffic_Vol"
        ]

    ].values

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(
        X
    )

    dbscan = DBSCAN(

        eps=0.5,

        min_samples=10

    )


    clusters = dbscan.fit_predict(
        X_scaled
    )


    temp_df[
        "DBSCAN_Cluster"
    ] = clusters

    dbscan_models[
        feature
    ] = {

        "model": dbscan,

        "scaler": scaler

    }

    cluster_summary = temp_df.groupby(

        "DBSCAN_Cluster"

    ).agg(

        Average_Traffic=(

            "Traffic_Vol",

            "mean"

        ),

        Maximum_Traffic=(

            "Traffic_Vol",

            "max"

        ),

        Minimum_Traffic=(

            "Traffic_Vol",

            "min"

        ),

        Records=(

            "Traffic_Vol",

            "count"

        )

    ).reset_index()


    feature_cluster_info[
        feature
    ] = cluster_summary


Training DBSCAN for: Hour

Training DBSCAN for: Day

Training DBSCAN for: Month

Training DBSCAN for: Year

Training DBSCAN for: Weekday

Training DBSCAN for: IsWeekend

Training DBSCAN for: IsHoliday

Training DBSCAN for: Weather

Training DBSCAN for: Weather_Desc

Training DBSCAN for: Temperature

Training DBSCAN for: Rainfall_last_hour

Training DBSCAN for: Snowfall_last_hour

Training DBSCAN for: Cloud_Cover


In [ ]:
trend_summaries = {}

for feature in analysis_features:

    print(
        f"Creating trend summary for: {feature}"
    )


    summary = train_df.groupby(

        feature,

        dropna=False

    ).agg(

        Average_Traffic_Volume=(

            "Traffic_Vol",

            "mean"

        ),

        Maximum_Traffic_Volume=(

            "Traffic_Vol",

            "max"

        ),

        Minimum_Traffic_Volume=(

            "Traffic_Vol",

            "min"

        ),

        Number_of_Records=(

            "Traffic_Vol",

            "count"

        )

    ).reset_index()


    summary[
        "Traffic_Level"
    ] = summary[
        "Average_Traffic_Volume"
    ].apply(
        get_traffic_level
    )


    trend_summaries[
        feature
    ] = summary



Creating trend summary for: Hour
Creating trend summary for: Day
Creating trend summary for: Month
Creating trend summary for: Year
Creating trend summary for: Weekday
Creating trend summary for: IsWeekend
Creating trend summary for: IsHoliday
Creating trend summary for: Weather
Creating trend summary for: Weather_Desc
Creating trend summary for: Temperature
Creating trend summary for: Rainfall_last_hour
Creating trend summary for: Snowfall_last_hour
Creating trend summary for: Cloud_Cover


In [ ]:
print("\nSaving bottleneck detection models...")


joblib.dump(

    dbscan_models,

    "saved_bottleneck_models/dbscan_models.pkl"

)


joblib.dump(

    feature_encodings,

    "saved_bottleneck_models/feature_encodings.pkl"

)


joblib.dump(

    feature_cluster_info,

    "saved_bottleneck_models/cluster_information.pkl"

)


joblib.dump(

    trend_summaries,

    "saved_bottleneck_models/trend_summaries.pkl"

)


# Save thresholds

traffic_thresholds = {

    "q33": q33,

    "q66": q66

}


joblib.dump(

    traffic_thresholds,

    "saved_bottleneck_models/traffic_thresholds.pkl"

)


# Save feature list

joblib.dump(

    analysis_features,

    "saved_bottleneck_models/analysis_features.pkl"

)


print("\n")
print("=" * 60)

print(
    "BOTTLENECK MODELS SAVED SUCCESSFULLY"
)

print("=" * 60)


print(
    "Saved Features:"
)

for feature in analysis_features:

    print(
        "-",
        feature
    )


Saving bottleneck detection models...


BOTTLENECK MODELS SAVED SUCCESSFULLY
Saved Features:
- Hour
- Day
- Month
- Year
- Weekday
- IsWeekend
- IsHoliday
- Weather
- Weather_Desc
- Temperature
- Rainfall_last_hour
- Snowfall_last_hour
- Cloud_Cover
